In [1]:
import torch
print(torch.cuda.is_available())  # True
print(torch.cuda.get_device_name(0))  # NVIDIA GeForce RTX 3050

True
NVIDIA GeForce RTX 3050 6GB Laptop GPU


In [2]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, random_split
from torchvision import datasets, transforms, models
from torchvision.models import DenseNet201_Weights, DenseNet169_Weights
import timm
from sklearn.metrics import accuracy_score, f1_score, classification_report
import os
from tqdm import tqdm  # Progress bars
import torch.onnx  # For ONNX export

In [3]:
# Transforms
train_transform = transforms.Compose([
    transforms.Resize(256),
    transforms.CenterCrop(224),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(10),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

val_transform = transforms.Compose([
    transforms.Resize(256),
    transforms.CenterCrop(224),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

# Load dataset
data_dir = '/home/rifat-cou/Documents/Project/Dataset_Raw'  # Your folder
full_dataset = datasets.ImageFolder(data_dir, transform=train_transform)
class_names = full_dataset.classes  # For classification_report

# 80/20 split
train_size = int(0.8 * len(full_dataset))
val_size = len(full_dataset) - train_size
train_dataset, val_dataset = random_split(full_dataset, [train_size, val_size])
val_dataset.dataset.transform = val_transform

# Loaders
batch_size = 16  # For 6GB VRAM; reduce if needed
train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True, num_workers=4, pin_memory=True)
val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False, num_workers=4, pin_memory=True)

num_classes = 5
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

print(device)
class_names = full_dataset.classes
print(f"Classes: {class_names}")
print(f"Train: {len(train_dataset)}, Val: {len(val_dataset)}")

cuda
Classes: ['Chikenpox', 'Cowpox', 'Measles', 'MonkeyPox', 'Normal']
Train: 2088, Val: 523


In [4]:
# DenseNet201 (peak at 300 epochs)
model1 = models.densenet201(weights=DenseNet201_Weights.DEFAULT)
model1.classifier = nn.Linear(model1.classifier.in_features, num_classes)
for param in model1.features[:8].parameters():
    param.requires_grad = False
model1 = model1.to(device)

# DenseNet169 (peak at 100 epochs)
model2 = models.densenet169(weights=DenseNet169_Weights.DEFAULT)
model2.classifier = nn.Linear(model2.classifier.in_features, num_classes)
for param in model2.features[:8].parameters():
    param.requires_grad = False
model2 = model2.to(device)

# EfficientNetB3 (peak at 300 epochs)
model3 = timm.create_model('efficientnet_b3', pretrained=True, num_classes=num_classes)
for param in model3.blocks[:4].parameters():
    param.requires_grad = False
model3 = model3.to(device)

In [5]:
def train_single_model(model, epochs, save_name):
    criterion = nn.CrossEntropyLoss()
    optimizer = optim.AdamW(model.parameters(), lr=1e-4)
    scheduler = optim.lr_scheduler.StepLR(optimizer, step_size=50, gamma=0.5)
    unfreeze_epoch = epochs // 2

    for epoch in range(epochs):
        model.train()
        train_loss = 0.0
        for inputs, labels in tqdm(train_loader):
            inputs, labels = inputs.to(device), labels.to(device)
            optimizer.zero_grad()
            outputs = model(inputs)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()
            train_loss += loss.item()
        
        # Validate
        model.eval()
        val_preds, val_labels = [], []
        with torch.no_grad():
            for inputs, labels in val_loader:
                inputs, labels = inputs.to(device), labels.to(device)
                outputs = model(inputs)
                preds = torch.argmax(outputs, dim=1)
                val_preds.extend(preds.cpu().numpy())
                val_labels.extend(labels.cpu().numpy())
        
        acc = accuracy_score(val_labels, val_preds)
        print(f'Epoch {epoch+1}/{epochs} - Loss: {train_loss/len(train_loader):.4f} - Val Acc: {acc:.4f}')
        
        scheduler.step()

        if epoch == unfreeze_epoch:
            for param in model.parameters():
                param.requires_grad = True
            optimizer = optim.AdamW(model.parameters(), lr=1e-5)
    
    # Save at peak (end)
    torch.save(model.state_dict(), f'{save_name}_peak.pth')

# Train each to peak
train_single_model(model1, 300, 'densenet201')
train_single_model(model2, 100, 'densenet169')
train_single_model(model3, 300, 'efficientnetb3')

100%|█████████████████████████████████████████| 131/131 [00:33<00:00,  3.94it/s]


Epoch 1/300 - Loss: 0.5345 - Val Acc: 0.8987


100%|█████████████████████████████████████████| 131/131 [00:32<00:00,  4.04it/s]


Epoch 2/300 - Loss: 0.1887 - Val Acc: 0.9197


100%|█████████████████████████████████████████| 131/131 [00:32<00:00,  4.03it/s]


...

Epoch 298/300 - Loss: 0.0001 - Val Acc: 0.9025


100%|█████████████████████████████████████████| 131/131 [00:29<00:00,  4.44it/s]


Epoch 299/300 - Loss: 0.0007 - Val Acc: 0.9101


100%|█████████████████████████████████████████| 131/131 [00:29<00:00,  4.44it/s]


Epoch 300/300 - Loss: 0.0006 - Val Acc: 0.9140


In [6]:
# Load peak models
model1.load_state_dict(torch.load('densenet201_peak.pth'))
model2.load_state_dict(torch.load('densenet169_peak.pth'))
model3.load_state_dict(torch.load('efficientnetb3_peak.pth'))

models_list = [model1, model2, model3]

def ensemble_predict(models, inputs):
    with torch.no_grad():
        probs = [torch.softmax(m(inputs), dim=1) for m in models]
        avg_probs = torch.mean(torch.stack(probs), dim=0)
    return torch.argmax(avg_probs, dim=1)

In [7]:
all_preds, all_labels = [], []
with torch.no_grad():
    for inputs, labels in val_loader:
        inputs, labels = inputs.to(device), labels.to(device)
        preds = ensemble_predict(models_list, inputs)
        all_preds.extend(preds.cpu().numpy())
        all_labels.extend(labels.cpu().numpy())

acc = accuracy_score(all_labels, all_preds)
f1 = f1_score(all_labels, all_preds, average='weighted')
print(classification_report(all_labels, all_preds, target_names=class_names))
print(f'Final Val Accuracy: {acc:.4f} - F1 Score: {f1:.4f}')

# Save
model_path = '/home/rifat-cou/Documents/Project/Ensemble Models'
ensemble_dir = os.path.join(model_path, 'V7_epoch_based')
os.makedirs(ensemble_dir, exist_ok=True)
torch.save(model1.state_dict(), os.path.join(ensemble_dir, 'densenet201.pth'))
torch.save(model2.state_dict(), os.path.join(ensemble_dir, 'densenet169.pth'))
torch.save(model3.state_dict(), os.path.join(ensemble_dir, 'efficientnetb3.pth'))

              precision    recall  f1-score   support

   Chikenpox       0.86      0.91      0.88       106
      Cowpox       0.97      0.96      0.97       107
     Measles       0.98      0.99      0.98        94
   MonkeyPox       0.90      0.85      0.88       108
      Normal       0.97      0.97      0.97       108

    accuracy                           0.93       523
   macro avg       0.94      0.94      0.94       523
weighted avg       0.94      0.93      0.93       523

Final Val Accuracy: 0.9350 - F1 Score: 0.9349


In [8]:
class EnsembleModel(nn.Module):
    def __init__(self, models):
        super().__init__()
        self.models = nn.ModuleList(models)

    def forward(self, x):
        probs = [torch.softmax(m(x), dim=1) for m in self.models]
        avg_probs = torch.mean(torch.stack(probs), dim=0)
        return avg_probs  # Softmax output for inference

ensemble_model = EnsembleModel(models_list).to('cpu')
dummy_input = torch.randn(1, 3, 224, 224)
onnx_path = os.path.join(ensemble_dir, 'epoch_based_ensemble.onnx')
torch.onnx.export(
    ensemble_model, 
    dummy_input, 
    onnx_path, 
    export_params=True, 
    opset_version=11,
    do_constant_folding=True, 
    input_names=['input'], 
    output_names=['output'], 
    dynamic_axes={'input': {0: 'batch_size'}, 'output': {0: 'batch_size'}}
)
print(f'Exported to ONNX: {onnx_path}')

Exported to ONNX: /home/rifat-cou/Documents/Project/Ensemble Models/V7_epoch_based/epoch_based_ensemble.onnx


In [9]:
!pip install onnxruntime

In [17]:
import onnxruntime as ort
import numpy as np
import torch

# Define your PyTorch model first
# For example:
model = torch.hub.load('pytorch/vision:v0.10.0', 'resnet18', pretrained=True)  # Example model
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = model.to(device)
model.eval()  # Set to evaluation mode

example_input = torch.randn(1, 3, 224, 224).to(device)
# Load ONNX
ort_session = ort.InferenceSession("/home/rifat-cou/Documents/Project/Ensemble Models/V7_epoch_based/epoch_based_ensemble.onnx")

# Prepare input (NumPy)
ort_input = {ort_session.get_inputs()[0].name: example_input.cpu().numpy()}

# Run inference
ort_output = ort_session.run(None, ort_input)[0]

# Get PyTorch output
pytorch_output = model(example_input).detach().cpu().numpy()

# Print shapes to understand the difference
print(f"PyTorch output shape: {pytorch_output.shape}")
print(f"ONNX output shape: {ort_output.shape}")

# Instead of comparing directly, you need to either:
# 1. Use a different PyTorch model that matches your ONNX model's output size
# 2. Or modify your PyTorch model to match the ONNX output
# 3. Or just verify the ONNX model works correctly without comparison

# For example, if your ONNX model is a modified version with 5 output classes:
# model = YourCustomModel(num_classes=5)  # Define a model with 5 output classes

print("ONNX model loaded successfully!")

Using cache found in /home/rifat-cou/.cache/torch/hub/pytorch_vision_v0.10.0


PyTorch output shape: (1, 1000)
ONNX output shape: (1, 5)
ONNX model loaded successfully!
